In [1]:
from libraries.inference_training import Configuration, ImageDataset
from libraries.inference_training import initCudaEnvironment, createTransforms
from libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from libraries.inference_training import trainModel, saveModel, loadModel
import random

In [2]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

In [3]:
def createModel(trainDirectory: str, testDirectory: str, modelName: str, epochs: int, labels: list[str], augment_data: bool, save_path: str, save_interval=0, maskdata=None , model_description="default"):
    """
    :param trainDirectory: path naar training dataset
    :param testDirectory: path naar testing dataset
    :param modelName: naam van model
    :param epochs: hoeveelheid epochs
    :param labels: list van labels, geef normaal ["parkeerplaatsen"] als er geen andere objecten zijn
    :param augment_data: bepaald of er image transforms gedaan worden, nog niet getest
    :param save_path: path naar save locatie van model
    :param maskdata: list van floats 
    :param save_interval: 
    :param model_description: beschrijft het model in de ONNX als het gesaved is
    :return:
    """
    if maskdata is None:
        maskdata = [0.2, 0.3, 0.5]
    config = Configuration()
    print("Device: " + str(config.device))
    config.setSaveInterval(save_interval)
    config.setSavePath(save_path)
    config.setIsCrowd(False)
    config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
    config.setFilePrefix("")
    config.setModelName(modelName)
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2 + 1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.setEpochs(epochs)
    config.setOnnxInfo(producer="Tygron", description=model_description)
    config.addLegendEntry("Background", 0, "#00000000")
    i = 1
    for label in labels:
        config.addLegendEntry(label, i, "#" + ''.join([random.choice('ABCDEF0123456789') for i in range(6)]))
        i += 1

    config.setOnnxMetaData(scoreThreshold=maskdata[0],
                           maskThreshold=maskdata[1],
                           strideFraction=maskdata[2])

    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    if augment_data:
        trainingDataset = ImageDataset(config, True, imageTransforms=createTransforms(True))
        testDataset = ImageDataset(config, False, createTransforms(False))
    else:
        trainingDataset = ImageDataset(config, True, createTransforms(False))
        testDataset = ImageDataset(config, False, createTransforms(False))

    print("Train Image count: " + str(trainingDataset.__len__()))
    print("Test Image count: " + str(testDataset.__len__()))

    if not trainingDataset.validateFiles():
        print("Inconsistent training dataset ")
        trainingDataset.validateFiles()

    if not testDataset.validateFiles():
        print("Inconsistent test dataset ")
        testDataset.validateFiles()

    print("Pytorch model name " + config.getPytorchModelFileName())
    print("Onnx file name " + config.getOnnxFileName())

    model = trainModel(config, trainingDataset, testDataset)
    model.eval()

    saveModel(config, model, epoch=epochs)

    exportOnnxModel(config, model)
    writeONNXMeta(config)

    return model, config

# create model template

In [ ]:
train_directory = "<insert train path here>"
test_directory = "<insert test path here>"
modelName = "<insert name here>"
epochs = 1
labels = ["<insert labels here>"]
augment = False
save_path = "<insert path to save location for model here>"
save_Interval = 0 # model is saved in between these amount of epochs
createModel(train_directory, test_directory, modelName, epochs, labels, augment, save_path, save_Interval)

# combo model with augment

In [4]:
train_directory = "C:/Users/Gebruiker/Desktop/homework/deep_learning_in_practice/datasets/combo_overlay_sets/train"
test_directory = "C:/Users/Gebruiker/Desktop/homework/deep_learning_in_practice/datasets/combo_overlay_sets/test"
modelName = "combo_model_with_new_augment"
epochs = 15
labels = ["parking_space"]  
augment = True
save_path = "C:/Users/Gebruiker/Desktop/homework/deep_learning_in_practice/models/combo_models/augment/"
save_Interval = 5
model, config = createModel(train_directory, test_directory, modelName, epochs, labels, augment, save_path, save_Interval)

Device: cuda
Train Image count: 1600
Test Image count: 800
Pytorch model name C:/Users/Gebruiker/Desktop/homework/deep_learning_in_practice/models/combo_models/augment/combo_model_with_new_augment
Onnx file name C:/Users/Gebruiker/Desktop/homework/deep_learning_in_practice/models/combo_models/augment/combo_model_with_new_augment.onnx


C:\Users\Gebruiker\Desktop\homework\deep_learning_in_practice\libraries\engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [  0/800]  eta: 0:47:07  lr: 0.000011  loss: 4.9868 (4.9868)  loss_classifier: 1.1407 (1.1407)  loss_box_reg: 0.0536 (0.0536)  loss_mask: 2.2781 (2.2781)  loss_objectness: 1.4078 (1.4078)  loss_rpn_box_reg: 0.1066 (0.1066)  time: 3.5339  data: 0.1023  max mem: 1658
Epoch: [0]  [ 10/800]  eta: 0:32:08  lr: 0.000074  loss: 4.5379 (4.4944)  loss_classifier: 1.0443 (1.0281)  loss_box_reg: 0.1083 (0.1077)  loss_mask: 1.8220 (1.6641)  loss_objectness: 1.3606 (1.5482)  loss_rpn_box_reg: 0.1066 (0.1463)  time: 2.4417  data: 0.0712  max mem: 1826
Epoch: [0]  [ 20/800]  eta: 0:31:04  lr: 0.000136  loss: 2.7876 (3.2561)  loss_classifier: 0.8215 (0.7923)  loss_box_reg: 0.1139 (0.1214)  loss_mask: 0.9695 (1.2374)  loss_objectness: 0.7484 (0.9932)  loss_rpn_box_reg: 0.0498 (0.1118)  time: 2.3336  data: 0.0654  max mem: 1827
Epoch: [0]  [ 30/800]  eta: 0:30:31  lr: 0.000199  loss: 1.6799 (2.7132)  loss_classifier: 0.3303 (0.6129)  loss_box_reg: 0.1139 (0.1262)  loss_mask: 0.7405 (1.1052) 

C:\Users\Gebruiker\miniconda3\envs\tygronai\Lib\site-packages\torch\nn\functional.py:4511: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  * torch.tensor(scale_factors[i], dtype=torch.float32)
C:\Users\Gebruiker\miniconda3\envs\tygronai\Lib\site-packages\torchvision\ops\boxes.py:166: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_x = torch.min(boxes_x, torch.tensor(width, dtype=boxes.dtype, device=boxes.device))
C:\Users\Gebruiker\miniconda3\envs\tygronai\Lib\site-packages\torchvision\ops\boxes.py:168: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sour

^ AP and AR converge to the results at the 15th epoch and don't change after with the current dataset.